# Gold: fact_order_reviews

## Import Helper Functions

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql
from src.gold.helper import get_changed_customer_ids, get_changed_order_ids
from src.gold.facts.order_reviews import build_fact_order_reviews, build_fact_order_reviews_incremental, validate_fact_order_reviews
from src.monitoring import write_dq_metrics
from src.watermark import get_last_commit_ts, get_effective_watermark
from src.writers import overwrite_table, replace_by_key

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
GOLD_NAMESPACE = cfg["general"]["namespaces"]["gold"]

cfg_order_reviews= cfg["gold"]["fact_order_reviews"]
SOURCE_TABLE = cfg_order_reviews["source_table"]
TARGET_TABLE = cfg_order_reviews["target_table"]
ORDERS_TABLE = cfg_order_reviews["orders_table"]
CUSTOMERS_TABLE = cfg_order_reviews["customers_table"]
DATE_TABLE = cfg_order_reviews["date_table"]
BUFFER_HOURS = cfg_order_reviews["buffer_hours"]
KEY_COLUMNS = cfg_order_reviews["key_columns"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["gold_order_reviews"])
        .getOrCreate()
)

## Run Pipeline

In [4]:
def run_fact_order_reviews_pipeline(spark):
    print("[START] fact_order_reviews pipeline")
    last_commit_ts = get_last_commit_ts(spark, TARGET_TABLE)
    print(f"[INFO] last_commit_ts = {last_commit_ts}")

    if last_commit_ts is None:
        print("[INFO] first run → full rebuild")
        df = build_fact_order_reviews(
            spark,
            SOURCE_TABLE,
            ORDERS_TABLE, 
            CUSTOMERS_TABLE,
            DATE_TABLE
        )
        metrics = validate_fact_order_reviews(df)
        write_dq_metrics(df, metrics, "fact_order_reviews", GOLD_NAMESPACE, "monitoring.dq_metrics")
        overwrite_table(df, TARGET_TABLE)

    effective_ts = get_effective_watermark(last_commit_ts, BUFFER_HOURS)
    changed_order_ids = get_changed_order_ids(spark, effective_ts)
    changed_customer_ids = get_changed_customer_ids(spark, effective_ts)

    if changed_customer_ids.isEmpty() and changed_order_ids.isEmpty():
        print("[INFO] no changes detected → skip")
        return

    print("[INFO] changes detected → incremental run")
    df = build_fact_order_reviews_incremental(
        spark,
        SOURCE_TABLE,
        ORDERS_TABLE, 
        CUSTOMERS_TABLE, 
        DATE_TABLE,
        changed_order_ids,
        changed_customer_ids,
    )
    metrics = validate_fact_order_reviews(df)
    write_dq_metrics(df, metrics, "fact_order_reviews", GOLD_NAMESPACE, "monitoring.dq_metrics")
    replace_by_key(spark, df, TARGET_TABLE, KEY_COLUMNS)
    print("[END] incremental update complete")

In [5]:
# if __name__ == "__main__":
#     from pyspark.sql import SparkSession

#     spark = SparkSession.builder.getOrCreate()
run_fact_order_reviews_pipeline(spark)

[START] fact_order_reviews pipeline
[INFO] last_commit_ts = None
[INFO] first run → full rebuild


[INFO] changes detected → incremental run


[END] incremental update complete


## Sanity Check

In [6]:
%%sql
SHOW TABLES IN polaris.gold;

+---------+-------------------+-----------+
|namespace|tableName          |isTemporary|
+---------+-------------------+-----------+
|gold     |fact_orders        |false      |
|gold     |dim_date           |false      |
|gold     |dim_customers_scd2 |false      |
|gold     |dim_sellers_scd2   |false      |
|gold     |dim_products_scd2  |false      |
|gold     |fact_order_items   |false      |
|gold     |fact_order_payments|false      |
|gold     |fact_order_reviews |false      |
+---------+-------------------+-----------+



In [7]:
%%sql
SELECT * FROM polaris.gold.fact_order_reviews
LIMIT 10

+--------------------------------+--------------------------------+------------+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+--------------------+---------------------+-----------------------+----------------------------------------------------------------+--------------------------------+
|review_id                       |order_id                        |review_score|review_comment_title  |review_comment_message                                                                                                                                                        |review_creation_date_sk|review_creation_date|review_answer_date_sk|review_answer_timestamp|customer_sk                                                     |customer_id                     |
+--------------------------------+------------------------------

In [8]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop() 